# Cluster Composition Data

In [35]:
import os
import re
import pandas as pd

# === Set your simulation base directory ===
base_dir = "../../Results/output"
output_csv = "../../Results/Data/Clusters/ClusterComposition.csv"

# === Columns to retain from each CSV file ===
columns_to_keep = [
    "MCS", "Jlf", "mu", "PP", "rep", "Cluster ID", "Leader Cells", "Follower Cells", "Total Cells",
    "Centroid_X", "Centroid_Y", "Member_Leader_X", "Member_Leader_Y",
    "Member_Follower_X", "Member_Follower_Y"
]

# === Default empty row for missing or empty files ===
def default_row(mcs, Jlf, mu, PP, rep):
    return {
        "MCS": mcs,
        "Jlf": float(Jlf),
        "mu": float(mu),
        "PP": float(PP),
        "rep": int(rep),
        "Cluster ID": 0,
        "Leader Cells": 0,
        "Follower Cells": 0,
        "Total Cells": 0,
        "Centroid_X": 0.0,
        "Centroid_Y": 0.0,
        "Member_Leader_X": 0.0,
        "Member_Leader_Y": 0.0,
        "Member_Follower_X": 0.0,
        "Member_Follower_Y": 0.0
    }

# === Regex patterns for folder and MCS parsing ===
folder_pattern = re.compile(r"scan_(\d+)_Jlf_([-.\d]+)_mu_([-.\d]+)_PP_([-.\d]+)_rep_(\d+)")
mcs_folder_pattern = re.compile(r"MCS_(\d+)")

# === Collect all valid dataframes ===
all_rows = []

for scan_folder in os.listdir(base_dir):
    scan_path = os.path.join(base_dir, scan_folder)
    if not os.path.isdir(scan_path):
        continue

    folder_match = folder_pattern.match(scan_folder)
    if not folder_match:
        continue

    scan_idx, Jlf, mu, PP, rep = folder_match.groups()
    position_data_folder = f"PositionData_{Jlf}_{mu}_{PP}"
    position_path = os.path.join(scan_path, position_data_folder)

    if not os.path.isdir(position_path):
        print(f"❌ Missing PositionData folder: {position_data_folder}")
        continue

    for mcs_folder in os.listdir(position_path):
        mcs_match = mcs_folder_pattern.match(mcs_folder)
        if not mcs_match:
            continue

        mcs_val = int(mcs_match.group(1))
        mcs_path = os.path.join(position_path, mcs_folder)
        if not os.path.isdir(mcs_path):
            continue

        filename = f"ClusterComposition_{Jlf}_{mu}_{PP}_MCS_{mcs_val}.csv"
        filepath = os.path.join(mcs_path, filename)

        if os.path.isfile(filepath):
            try:
                df = pd.read_csv(filepath)
                if df.empty:
                    all_rows.append(pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)]))
                else:
                    df["MCS"] = mcs_val
                    df["Jlf"] = float(Jlf)
                    df["mu"] = float(mu)
                    df["PP"] = float(PP)
                    df["rep"] = int(rep)
                    df = df[columns_to_keep]
                    all_rows.append(df)
            except Exception as e:
                print(f"❌ Error reading {filepath}: {e}")
                all_rows.append(pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)]))
        else:
            print(f"⚠️ File missing: {filepath}")
            all_rows.append(pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)]))

# === Combine and export ===
if all_rows:
    combined_df = pd.concat(all_rows, ignore_index=True)

    if not combined_df.empty:
        combined_df.sort_values(by=["Jlf", "mu", "PP", "rep", "MCS" ], inplace=True)
        os.makedirs(os.path.dirname(output_csv), exist_ok=True)
        combined_df.to_csv(output_csv, index=False)

        print(f"\n✅ Saved sorted cluster data to:\n{output_csv}")
        print(f"🟢 Rows: {len(combined_df)}, Unique simulations: {combined_df[['Jlf', 'mu', 'PP', 'rep']].drop_duplicates().shape[0]}")
    else:
        print("\n❗Cluster files were collected, but all were empty.")
else:
    print("\n❗No cluster composition files were processed.")



✅ Saved sorted cluster data to:
../../Results/Data/Clusters/ClusterComposition.csv
🟢 Rows: 1143231, Unique simulations: 13310


In [4]:
import pandas as pd
import numpy as np

# === Paths ===
full_csv_path = "../../Results/Data/Clusters/ClusterComposition.csv"
summary_csv_path = "../../Results/Data/Clusters/ClusterComposition_GroupStats.csv"
summary_avg_csv_path = "../../Results/Data/Clusters/ClusterComposition_GroupStats_AvgAcrossReps.csv"

# === Load full data ===
df_all = pd.read_csv(full_csv_path)

# === Extract all combinations of parameter, replicate, and MCS ===
group_cols = ["Jlf", "mu", "PP", "rep", "MCS"]
all_combinations = df_all[group_cols].drop_duplicates()

# === Filter to real clusters (Total Cells > 0) for stat computation ===
df_clusters = df_all[df_all["Total Cells"] > 0].copy()

# === Group stats on real clusters only ===
stats = df_clusters.groupby(group_cols).agg(
    Number_of_Clusters=('Cluster ID', 'nunique'),
    Total_Cells=('Total Cells', 'sum'),
    Total_Leader_Cells=('Leader Cells', 'sum'),
    Total_Follower_Cells=('Follower Cells', 'sum'),
    Avg_Cluster_Size=('Total Cells', 'mean'),
    Median_Cluster_Size=('Total Cells', 'median'),
    Avg_Leader_Cells_Per_Cluster=('Leader Cells', 'mean'),
    Avg_Follower_Cells_Per_Cluster=('Follower Cells', 'mean'),
    Max_Cluster_Size=('Total Cells', 'max'),
    Min_Cluster_Size=('Total Cells', 'min'),
    Std_Cluster_Size=('Total Cells', 'std')
).reset_index()

# === Add percentage of leader cells
stats["Percent_Leader_Cells"] = stats.apply(
    lambda row: (row["Total_Leader_Cells"] / row["Total_Cells"]) * 100
    if row["Total_Cells"] > 0 else 0,
    axis=1
)

# === Merge with full combination list to ensure even 0-cluster cases are included ===
summary = pd.merge(all_combinations, stats, on=group_cols, how='left')

# === Fill NaN with 0 where appropriate ===
fill_zero_cols = [
    "Number_of_Clusters", "Total_Cells", "Total_Leader_Cells", "Total_Follower_Cells",
    "Avg_Cluster_Size", "Median_Cluster_Size", "Avg_Leader_Cells_Per_Cluster",
    "Avg_Follower_Cells_Per_Cluster", "Max_Cluster_Size", "Min_Cluster_Size",
    "Std_Cluster_Size", "Percent_Leader_Cells"
]
summary[fill_zero_cols] = summary[fill_zero_cols].fillna(0)

# === Save per-replicate group stats
summary.sort_values(by=group_cols).to_csv(summary_csv_path, index=False)
print(f"\n📊 Final group-level summary (including 0-cluster cases) saved to:\n{summary_csv_path}")
print(f"🟢 Rows: {len(summary)}, Unique parameter–rep–MCS groups: {summary[group_cols].nunique()}")

# === Compute mean and std across replicates ===
avg_group_cols = ["Jlf", "mu", "PP", "MCS"]
metrics_to_avg = [col for col in summary.columns if col not in group_cols]

# Mean
summary_avg = summary.groupby(avg_group_cols)[metrics_to_avg].mean().reset_index()

# Std
summary_std = summary.groupby(avg_group_cols)[metrics_to_avg].std().reset_index()
summary_std = summary_std.add_suffix("_std")
summary_std.rename(columns={col+"_std": col for col in avg_group_cols}, inplace=True)

# === Merge mean and std into one DataFrame
summary_combined = pd.merge(summary_avg, summary_std, on=avg_group_cols, how='left')

# === Round integer-like columns to nearest whole number

# Round cluster counts normally
summary_combined["Number_of_Clusters"] = summary_combined["Number_of_Clusters"].round(0).astype(int)

# Round leader and follower cells
summary_combined["Total_Leader_Cells"] = summary_combined["Total_Leader_Cells"].round(0).astype(int)
summary_combined["Total_Follower_Cells"] = summary_combined["Total_Follower_Cells"].round(0).astype(int)

# Ensure Total_Cells = Leader + Follower
summary_combined["Total_Cells"] = summary_combined["Total_Leader_Cells"] + summary_combined["Total_Follower_Cells"]

int_cols = ["Number_of_Clusters", "Total_Cells", "Total_Leader_Cells", "Total_Follower_Cells"]

# === Round remaining float columns to 3 decimal places
exclude_cols = set(avg_group_cols + int_cols)
float_cols_to_round = [col for col in summary_combined.columns if col not in exclude_cols and summary_combined[col].dtype in [np.float32, np.float64, np.float64]]

summary_combined[float_cols_to_round] = summary_combined[float_cols_to_round].round(3)


# === Filter only cases with at least 1 cluster
summary_combined = summary_combined[summary_combined["Number_of_Clusters"] > 0].copy()


# === Save final averaged and rounded summary
summary_combined.to_csv(summary_avg_csv_path, index=False)
print(f"\n📈 Replicate-averaged summary saved to:\n{summary_avg_csv_path}")
print(f"🟢 Rows: {len(summary_combined)}, Unique parameter–MCS combinations: {summary_combined[['Jlf', 'mu', 'PP']].drop_duplicates().shape[0]}")


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_80865/1494166503.py:10: DtypeWarning: Columns (11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv(full_csv_path)



📊 Final group-level summary (including 0-cluster cases) saved to:
../../Results/Data/Clusters/ClusterComposition_GroupStats.csv
🟢 Rows: 944113, Unique parameter–rep–MCS groups: Jlf    11
mu     11
PP     11
rep    10
MCS    71
dtype: int64

📈 Replicate-averaged summary saved to:
../../Results/Data/Clusters/ClusterComposition_GroupStats_AvgAcrossReps.csv
🟢 Rows: 10204, Unique parameter–MCS combinations: 418


### txt and json

In [24]:
import pandas as pd
import json
import os

# === Input CSV (already generated by your cluster script) ===
input_csv = "../../Results/Data/Clusters/ClusterComposition.csv"

# === Output Paths ===
output_json_path = "../../Results/Data/Clusters/ClusterComposition_Summary.json"
output_txt_path = "../../Results/Data/Clusters/ClusterComposition_Summary.txt"

# === Load the full cluster data ===
df = pd.read_csv(input_csv)

# === Filter: Keep only rows where at least one cluster exists ===
df_valid = df[df["Total Cells"] > 0].copy()

# === Build nested dictionary ===
nested_data = {}

for _, row in df_valid.iterrows():
    # Build hierarchical keys
    param_key = f"Jlf_{row['Jlf']}_mu_{row['mu']}_PP_{row['PP']}"
    rep_key = f"rep_{int(row['rep'])}"
    mcs_key = f"MCS_{int(row['MCS'])}"
    cluster_key = f"cluster_{int(row['Cluster ID'])}"

    # Cluster-level data excluding hierarchy keys
    cluster_data = {k: row[k] for k in row.index if k not in ['Jlf', 'mu', 'PP', 'rep', 'MCS', 'Cluster ID']}

    # Add to nested structure
    nested_data.setdefault(param_key, {}) \
               .setdefault(rep_key, {}) \
               .setdefault(mcs_key, {})[cluster_key] = cluster_data

# === Save JSON ===
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)
with open(output_json_path, 'w') as f_json:
    json.dump(nested_data, f_json, indent=2)

print(f"✅ JSON summary saved to: {output_json_path}")

# === Save human-readable TXT ===
with open(output_txt_path, 'w') as f_txt:
    for param_key, reps in nested_data.items():
        f_txt.write(f"\n=== Parameter: {param_key} ===\n")
        for rep_key, mcs_dict in reps.items():
            f_txt.write(f"  >> {rep_key}\n")
            for mcs_key, clusters in mcs_dict.items():
                f_txt.write(f"    [{mcs_key}]\n")
                for cluster_key, data in clusters.items():
                    f_txt.write(f"      - {cluster_key}: {data}\n")

print(f"✅ TXT summary saved to: {output_txt_path}")


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_49207/3026097599.py:13: DtypeWarning: Columns (11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


✅ JSON summary saved to: ../../Results/Data/Clusters/ClusterComposition_Summary.json
✅ TXT summary saved to: ../../Results/Data/Clusters/ClusterComposition_Summary.txt


# Cluster Analysis

### Cluster Rate

The **cluster rate** is defined as the discrete difference of the cluster count over time. It quantifies the net change in the number of clusters between two consecutive time steps:

$$
\text{Cluster Rate}_{t} = \Delta C_t = C_{t} - C_{t-1}
$$

Or, in approximate derivative form:

$$
\frac{dC}{dt} \approx \frac{C(t+\Delta t) - C(t)}{\Delta t}
$$

This metric can be:
- **Positive** (formation of new clusters),
- **Negative** (merging or disappearance),
- **Zero** (stable cluster count).

---

### Cluster Formation Rate

The **cluster formation rate** captures only the positive changes (i.e., formation of new clusters):

$$
\text{CFR}(t) = \max(0, \Delta C_t) = \max\left(0, \frac{C(t+\Delta t) - C(t)}{\Delta t}\right)
$$

This reflects the number of clusters that newly emerged between  $t-1$  and  $t $.


In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# -------------------------------
# STEP 1: Load the simulation data
# -------------------------------
file_path = "../../Results/Data/Clusters/ClusterComposition.csv"
df = pd.read_csv(file_path, low_memory=False)

# -------------------------------------------
# STEP 2: Compute Cluster Rate and CFR per replicate
# -------------------------------------------
group_cols = ['Jlf', 'mu', 'PP', 'rep']  # Unique simulation setup
cluster_rate_records = []

# Loop through each replicate of each parameter combination
for group_key, group_df in df.groupby(group_cols):
    # Count number of unique clusters at each time step (MCS)
    cluster_count_per_mcs = group_df.groupby('MCS')['Cluster ID'].nunique().sort_index()
    
    # Compute Cluster Rate: ΔCₜ = Cₜ - Cₜ₋₁
    delta_ct = cluster_count_per_mcs.diff().fillna(0).astype(int)
    
    # Compute CFR: max(0, ΔCₜ)
    cfr_t = delta_ct.clip(lower=0)
    
    # Store results
    for mcs, rate in delta_ct.items():
        cluster_rate_records.append({
            'Jlf': group_key[0],
            'mu': group_key[1],
            'PP': group_key[2],
            'rep': group_key[3],
            'MCS': mcs,
            'Cluster_Rate': rate,
            'CFR': cfr_t.loc[mcs]
        })

# Convert to DataFrame
cluster_rate_df = pd.DataFrame(cluster_rate_records)

# Save per-replicate Cluster Rate and CFR
output_csv_path = "../../Results/Cluster/ClusterRate_CFR.csv"
cluster_rate_df.to_csv(output_csv_path, index=False)

# ----------------------------------------------------
# STEP 3: Average Cluster Rate and CFR across replicates
# ----------------------------------------------------
avg_cluster_rate_df = (
    cluster_rate_df
    .groupby(['Jlf', 'mu', 'PP', 'MCS'])[['Cluster_Rate', 'CFR']]
    .mean()
    .reset_index()
)

# Save averaged results
avg_output_csv_path = "../../Results/Cluster/Average_ClusterRate_CFR.csv"
avg_cluster_rate_df.to_csv(avg_output_csv_path, index=False)

# ----------------------------------------------------
# STEP 4: Select specific parameter set to visualize
# ----------------------------------------------------
selected_params = (2.0, 27.0, 0.4)  # Choose one (Jlf, mu, PP) combo

replicate_df = cluster_rate_df[
    (cluster_rate_df['Jlf'] == selected_params[0]) &
    (cluster_rate_df['mu'] == selected_params[1]) &
    (cluster_rate_df['PP'] == selected_params[2])
]

avg_df = avg_cluster_rate_df[
    (avg_cluster_rate_df['Jlf'] == selected_params[0]) &
    (avg_cluster_rate_df['mu'] == selected_params[1]) &
    (avg_cluster_rate_df['PP'] == selected_params[2])
]

# ----------------------------------------------------
# STEP 5: Plot all replicates (each replicate in its own subplot)
# ----------------------------------------------------
rep_ids = replicate_df['rep'].unique()
num_reps = len(rep_ids)
cols = min(5, num_reps)  
rows = math.ceil(num_reps / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), sharex=True, sharey=True)
axes = axes.flatten() if num_reps > 1 else [axes]

for i, rep_id in enumerate(rep_ids):
    rep_data = replicate_df[replicate_df['rep'] == rep_id]
    axes[i].plot(rep_data['MCS'], rep_data['Cluster_Rate'], label='Cluster Rate', marker='o', markersize=3)
    axes[i].plot(rep_data['MCS'], rep_data['CFR'], label='CFR', marker='s', markersize=3)
    axes[i].set_title(f"Replicate {rep_id}")
    axes[i].grid(True)
    axes[i].legend(fontsize="small")


for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'Cluster Rate and CFR per Replicate\n(Jlf={selected_params[0]}, mu={selected_params[1]}, PP={selected_params[2]})', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save replicate subplot grid
rep_plot_path = "../../Results/Cluster/ClusterRate_CFR.png"
plt.savefig(rep_plot_path)
plt.close()

# ----------------------------------------------------
# STEP 6: Plot average Cluster Rate and CFR across replicates
# ----------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(avg_df['MCS'], avg_df['Cluster_Rate'], label='Avg Cluster Rate (ΔCₜ)', marker='o', markersize=4)
plt.plot(avg_df['MCS'], avg_df['CFR'], label='Avg CFR', marker='s', markersize=4)
plt.title(f'Average Cluster Dynamics\nJlf={selected_params[0]}, mu={selected_params[1]}, PP={selected_params[2]}')
plt.xlabel('MCS')
plt.ylabel('Rate')
plt.grid(True)
plt.legend()
plt.tight_layout()

# Save average plot
avg_plot_path = "../../Results/Cluster/Average_ClusterRate_CFR.png"
plt.savefig(avg_plot_path)
plt.close()

# ----------------------------------------------------
# STEP 7: Final output confirmation
# ----------------------------------------------------
print("✔ Cluster Rate (per replicate) saved to:", output_csv_path)
print("✔ Average Cluster Rate (across replicates) saved to:", avg_output_csv_path)
print("✔ Replicate plots saved to:", rep_plot_path)
print("✔ Average plot saved to:", avg_plot_path)


✔ Cluster Rate (per replicate) saved to: ../../Results/Cluster/ClusterRate_CFR.csv
✔ Average Cluster Rate (across replicates) saved to: ../../Results/Cluster/Average_ClusterRate_CFR.csv
✔ Replicate plots saved to: ../../Results/Cluster/ClusterRate_CFR.png
✔ Average plot saved to: ../../Results/Cluster/Average_ClusterRate_CFR.png


### Cluster Stability and Deformation

To quantify the stability of cluster sizes over time, we use the **Coefficient of Variation (CV)** of average cluster size. It measures the variation in cluster size over time. Let $S_i(t)$  be the size (number of cells) of cluster  $i$  at time  $t $, and $\overline{S}_t$  be the mean size of clusters at time $t$ .

- **Variance of cluster sizes over time:**

$$
\sigma_S^2(t) = \frac{1}{N_t} \sum_{i=1}^{N_t} \left(S_i(t) - \overline{S}_t\right)^2
$$

- **Cluster Size Coefficient of Variation (CV):**

$$
\text{CV}_{\overline{S}_t} = \frac{\sigma_S(t)}{\overline{S}_t}
$$

A high CV suggests instability in cluster size reflecting greater heterogeneity, fragmentation, or active invasion., while a low CV indicates stable, uniform cluster structures.
A **CV close to 0** indicates uniform, stable clusters, typically reflecting cohesive tumor growth. A **CV between 0.1 and 0.3** suggests mild to moderate structural heterogeneity, often due to asynchronous growth or early invasion. A **CV above 0.4** indicates significant variability in cluster sizes, which may correspond to fragmentation, leader–follower dynamics, or sustained invasive behavior. Therefore, rising CV over time can signal a structural transition from uniform expansion to multimodal or unstable tumor morphology.



In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os

# ========== File Paths ==========
input_file = "../../Results/Data/Clusters/ClusterComposition.csv"
cv_per_cluster_path = "../../Results/Cluster/ClusterSizeCV_perCluster.csv"
cv_per_rep_path = "../../Results/Cluster/ClusterSizeCV.csv"
cv_avg_path = "../../Results/Cluster/ClusterSizeCV_Avg.csv"
cv_plot_path = "../../Results/Cluster/ClusterCV.png"
cv_avg_plot_path = "../../Results/Cluster/Avg_ClusterCV.png"

# Ensure output directory exists
os.makedirs("../../Results/Cluster", exist_ok=True)

# ========== Load Data ==========
df = pd.read_csv(input_file, low_memory=False)

# ========== 1. Compute CV per Cluster ID ==========
group_cols_cluster = ['Jlf', 'mu', 'PP', 'rep', 'Cluster ID']
cv_cluster_results = []

for (jlf, mu, pp, rep, cluster_id), cluster_df in df.groupby(group_cols_cluster):
    total_cells_over_time = cluster_df.sort_values('MCS')['Total Cells'].values
    mean_size = np.mean(total_cells_over_time)
    std_size = np.std(total_cells_over_time)
    cv = std_size / mean_size if mean_size > 0 else np.inf

    cv_cluster_results.append({
        'Jlf': jlf,
        'mu': mu,
        'PP': pp,
        'rep': rep,
        'Cluster ID': cluster_id,
        'CV': cv if np.isfinite(cv) else '-',
        'Mean Size': mean_size,
        'Std Dev Size': std_size,
        'Time Points': len(total_cells_over_time)
    })

cv_cluster_df = pd.DataFrame(cv_cluster_results)
cv_cluster_df.to_csv(cv_per_cluster_path, index=False)
print(f"✔ Saved CV per Cluster to {cv_per_cluster_path}")

# ========== 2. Compute CV per Replicate per MCS ==========
group_cols_mcs = ['Jlf', 'mu', 'PP', 'rep', 'MCS']
cv_results = []

for (jlf, mu, pp, rep, mcs), group_df in df.groupby(group_cols_mcs):
    cluster_sizes = group_df['Total Cells'].values
    cluster_sizes = cluster_sizes[cluster_sizes > 0]

    if len(cluster_sizes) == 0 or np.mean(cluster_sizes) == 0:
        mean_size = std_size = count = 0
        cv = np.inf
    else:
        mean_size = np.mean(cluster_sizes)
        std_size = np.std(cluster_sizes)
        cv = std_size / mean_size
        count = len(cluster_sizes)

    cv_results.append({
        'Jlf': jlf,
        'mu': mu,
        'PP': pp,
        'rep': rep,
        'MCS': mcs,
        'Cluster Count': count,
        'Mean Size': mean_size,
        'Std Dev': std_size,
        'CV': cv if np.isfinite(cv) else 0
    })

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(cv_per_rep_path, index=False)
print(f"✔ Saved CV per MCS per Replicate to {cv_per_rep_path}")

# ========== 3. Compute Average CV across Replicates ==========
avg_cv_df = cv_df.groupby(['Jlf', 'mu', 'PP', 'MCS']).agg({
    'Cluster Count': 'mean',
    'Mean Size': 'mean',
    'Std Dev': 'mean',
    'CV': 'mean'
}).reset_index()

avg_cv_df.rename(columns={
    'Cluster Count': 'Avg Cluster Count',
    'Mean Size': 'Avg Mean Size',
    'Std Dev': 'Avg Std Dev',
    'CV': 'Avg CV'
}, inplace=True)

avg_cv_df.to_csv(cv_avg_path, index=False)
print(f"✔ Saved Averaged CV across Replicates to {cv_avg_path}")

# ========== 4. Plotting for a Parameter Combination ==========
plot_jlf = 3
plot_mu = 27
plot_pp = 0.4

# --- Replicate Subplots ---
subset_reps = cv_df[
    (cv_df['Jlf'] == plot_jlf) &
    (cv_df['mu'] == plot_mu) &
    (cv_df['PP'] == plot_pp)
]
replicates = sorted(subset_reps['rep'].unique())
num_reps = len(replicates)

if num_reps > 0:
    n_cols = min(4, num_reps)
    n_rows = math.ceil(num_reps / n_cols)
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharey=True)
    axs = axs.flatten()

    for i, rep in enumerate(replicates):
        rep_data = subset_reps[subset_reps['rep'] == rep].sort_values("MCS")
        axs[i].plot(rep_data["MCS"], rep_data["CV"], label=f"Rep {rep}")
        axs[i].set_title(f"Rep {rep}")
        axs[i].set_xlabel("MCS")
        axs[i].set_ylabel("CV of Cluster Size")
        axs[i].grid(True)

    for j in range(i + 1, len(axs)):
        axs[j].axis("off")

    plt.suptitle(f"CV per Replicate for Jlf={plot_jlf}, mu={plot_mu}, PP={plot_pp}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(cv_plot_path)
    plt.close()
    print(f"✔ Saved CV replicate subplot to {cv_plot_path}")
else:
    print(f"⚠ No replicates found for Jlf={plot_jlf}, mu={plot_mu}, PP={plot_pp}")

# --- Average Line Plot ---
avg_subset = avg_cv_df[
    (avg_cv_df['Jlf'] == plot_jlf) &
    (avg_cv_df['mu'] == plot_mu) &
    (avg_cv_df['PP'] == plot_pp)
].sort_values("MCS")

if not avg_subset.empty:
    plt.figure(figsize=(7, 5))
    plt.plot(avg_subset["MCS"], avg_subset["Avg CV"], color='red', linewidth=2)
    plt.title(f"Averaged CV Across Replicates\nJlf={plot_jlf}, mu={plot_mu}, PP={plot_pp}")
    plt.xlabel("MCS")
    plt.ylabel("Avg CV of Cluster Size")
    plt.grid(True)
    plt.savefig(cv_avg_plot_path)
    plt.close()
    print(f"✔ Saved average CV plot to {cv_avg_plot_path}")
else:
    print(f"⚠ No averaged data found for Jlf={plot_jlf}, mu={plot_mu}, PP={plot_pp}")


✔ Saved CV per Cluster to ../../Results/Cluster/ClusterSizeCV_perCluster.csv
✔ Saved CV per MCS per Replicate to ../../Results/Cluster/ClusterSizeCV.csv
✔ Saved Averaged CV across Replicates to ../../Results/Cluster/ClusterSizeCV_Avg.csv
✔ Saved CV replicate subplot to ../../Results/Cluster/ClusterCV.png
✔ Saved average CV plot to ../../Results/Cluster/Avg_ClusterCV.png


## Phenotypic Composition Analysis

### Leader Cell Percentage

At each time step, the proportion of leader cells is defined as the **leader cell percentage** of a cluster at time  $t $:

$$
p_L(t) = \frac{L_{i}^{(t)}}{L_{i}^{(t)} + F_{i}^{(t)}} \times 100
$$

Clusters or time windows can be classified based on $ p_L(t) $:

- **Leader-dominated**:  $p_L(t) > \theta $
- **Follower-dominated**:  $p_L(t) \leq \theta$ 

where  $\theta$  is a defined threshold (e.g.,  $\theta = 60\% $).

---

### Composition-Based Stability

Let $C^{\text{LD}}_t$  and  $C^{\text{FD}}_t$  represent the number of leader- and follower-dominated clusters at time  $t $. We compute separate CV values for each:

$$
\text{CV}^{\text{LD}} = \frac{\sigma^{\text{LD}}(t)}{\overline{S}^{\text{LD}}_t}, \quad
\text{CV}^{\text{FD}} = \frac{\sigma^{\text{FD}}(t)}{\overline{S}^{\text{FD}}_t}
$$

Comparing  $\text{CV}^{\text{LD}}$  and $\text{CV}^{\text{FD}}$  helps assess which type exhibits more stable sizes.

---

### Phenotypic Index and Entropy

For a cluster  $i$  at time  $t$ , let  $L_i(t)$  be the number of leader cells and  $F_i(t) $ the number of follower cells.

- **Phenotypic Index:**

$$
PI_i(t) = \frac{|L_{i}^{(t)} - F_{i}^{(t)}|}{L_{i}^{(t)} + F_{i}^{(t)}} = \frac{|L_{i}^{(t)} - F_{i}^{(t)}|}{T^t_i}
$$

- **Shannon Entropy of Composition:**

$$
p_L = \frac{L_{i}^{(t)} }{T^t_i}, \quad p_F = \frac{F_{i}^{(t)} }{T^t_i}
$$

$$
H_i(t) = -p_L \log_2(p_L) - p_F \log_2(p_F)
$$

Entropy quantifies phenotypic diversity, with higher values indicating more balanced compositions.
The Phenotypic Index ($PI_i(t)$) measures the imbalance between leader and follower cells within a cluster, ranging from 0 (perfectly balanced) to 1 (entirely one type). A high $PI$ suggests strong phenotypic skew, which may signal specialization or dominance. In contrast, the Entropy $H_i(t)$ captures the diversity in phenotypic makeup, peaking at 1 when leaders and followers are equally represented ($p_L = p_F = 0.5$) and dropping to 0 when one phenotype dominates completely. Thus, together, these metrics provide insight into the heterogeneity and cooperative balance of tumor cell clusters, which can influence collective behavior and invasive potential.

In [ ]:
import pandas as pd
import numpy as np

file_path = "../../Results/Data/Clusters/ClusterComposition.csv"
df = pd.read_csv(file_path, low_memory=False)

df["Leader Cells"] = pd.to_numeric(df["Leader Cells"], errors='coerce')
df["Follower Cells"] = pd.to_numeric(df["Follower Cells"], errors='coerce')
df["Total Cells"] = pd.to_numeric(df["Total Cells"], errors='coerce')


L = df["Leader Cells"]        # Number of leader cells
F = df["Follower Cells"]      # Number of follower cells
T = L + F                     # Total cells = Leader + Follower

# ------------------------------------------------------
#  Leader Percentage
# Formula: p_L(t) = (L / (L + F)) * 100
# ------------------------------------------------------
threshold = 60  # Classification threshold in percent
df["Leader_Percentage"] = (L / T.replace(0, np.nan)) * 100

# ------------------------------------------------------
#  Dominance Classification
# If leader percentage > threshold => Leader-dominated
# Else => Follower-dominated
# ------------------------------------------------------
df["Dominance"] = np.where(df["Leader_Percentage"] > threshold, "Leader-dominated", "Follower-dominated")
df.loc[T == 0, "Dominance"] = np.nan  # Handle cases where total cells = 0


# ------------------------------------------------------
# Phenotypic Index
# Formula: PI_i(t) = |L - F| / (L + F)
# Measures degree of phenotypic imbalance
# ------------------------------------------------------
df["Phenotypic_Index"] = np.abs(L - F) / T.replace(0, np.nan)

# ------------------------------------------------------
# Shannon Entropy
# Formula: H = -p_L * log2(p_L) - p_F * log2(p_F)
# Measures phenotypic diversity (max = 1 when L=F, min = 0 when one type dominates)
# ------------------------------------------------------
p_L = L / T.replace(0, np.nan)
p_F = F / T.replace(0, np.nan)
df["Entropy"] = -(
    p_L * np.log2(p_L.replace(0, np.nan)) +
    p_F * np.log2(p_F.replace(0, np.nan))
)

result_df = df[[
    "Jlf", "mu", "PP", "rep", "MCS", "Cluster ID",
    "Leader_Percentage", "Dominance", "Phenotypic_Index", "Entropy"
]]

# Save to CSV
output_path =  "../../Results/Cluster/PhenotypicComposition.csv"
result_df.to_csv(output_path, index=False)

output_path


'../../Results/Cluster/PhenotypicComposition.csv'

In [42]:
import pandas as pd
import numpy as np

# Load original composition file (ensure it includes Total Cells and Dominance)
file_path = "../../Results/Data/Clusters/ClusterComposition.csv"
df = pd.read_csv(file_path)

# Ensure numeric types
df["Leader Cells"] = pd.to_numeric(df["Leader Cells"], errors='coerce')
df["Follower Cells"] = pd.to_numeric(df["Follower Cells"], errors='coerce')
df["Total Cells"] = pd.to_numeric(df["Total Cells"], errors='coerce')

# Recompute dominance if not already in the file
L = df["Leader Cells"]
F = df["Follower Cells"]
T = L + F
threshold = 60
df["Leader_Percentage"] = (L / T.replace(0, np.nan)) * 100
df["Dominance"] = np.where(df["Leader_Percentage"] > threshold, "Leader-dominated", "Follower-dominated")
df.loc[T == 0, "Dominance"] = np.nan  # Skip zero-sized clusters

# --------------------------------------------------
# Compute CV of size over time for LD and FD clusters
# Grouping: each cluster over time in a given sim run
# --------------------------------------------------
group_cols = ["Jlf", "mu", "PP", "rep", "Cluster ID", "Dominance"]
cv_records = []

for group_key, cluster_df in df.groupby(group_cols):
    jlf, mu, pp, rep, cluster_id, dom_type = group_key
    cluster_sizes = cluster_df["Total Cells"].dropna().values

    if len(cluster_sizes) == 0 or np.mean(cluster_sizes) == 0:
        cv = np.inf
    else:
        mean_size = np.mean(cluster_sizes)
        std_size = np.std(cluster_sizes)
        cv = std_size / mean_size

    cv_records.append({
        "Jlf": jlf,
        "mu": mu,
        "PP": pp,
        "rep": rep,
        "Dominance": dom_type,
        "CV": cv if np.isfinite(cv) else 0,
        "Mean Size": mean_size if len(cluster_sizes) > 0 else 0,
        "Std Size": std_size if len(cluster_sizes) > 0 else 0,
        "Cluster Count": len(cluster_sizes),
    })

# Convert to DataFrame
cv_df = pd.DataFrame(cv_records)

# --------------------------------------------------
# Aggregate CVs across clusters per dominance group
# Result: one LD and one FD CV per sim run
# --------------------------------------------------
stability_summary = cv_df.groupby(["Jlf", "mu", "PP", "rep", "Dominance"]).agg({
    "CV": "mean",
    "Mean Size": "mean",
    "Std Size": "mean"
}).reset_index()

# Optional pivot for easier comparison
pivot_df = stability_summary.pivot(
    index=["Jlf", "mu", "PP", "rep"],
    columns="Dominance",
    values="CV"
).reset_index()

# Save to file
output_path = "../../Results/Cluster/ClusterStability_by_Dominance.csv"
stability_summary.to_csv(output_path, index=False)

print(f"Saved dominance-based cluster stability to: {output_path}")


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_80865/2118834574.py:6: DtypeWarning: Columns (11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Saved dominance-based cluster stability to: ../../Results/Cluster/ClusterStability_by_Dominance.csv


In [44]:
import pandas as pd
import numpy as np

# Load composition data
file_path = "../../Results/Data/Clusters/ClusterComposition.csv"
df = pd.read_csv(file_path)

# Convert to numeric
df["Leader Cells"] = pd.to_numeric(df["Leader Cells"], errors='coerce')
df["Follower Cells"] = pd.to_numeric(df["Follower Cells"], errors='coerce')
df["Total Cells"] = pd.to_numeric(df["Total Cells"], errors='coerce')

# Recalculate dominance
L = df["Leader Cells"]
F = df["Follower Cells"]
T = L + F
threshold = 60
df["Leader_Percentage"] = (L / T.replace(0, np.nan)) * 100
df["Dominance"] = np.where(df["Leader_Percentage"] > threshold, "Leader-dominated", "Follower-dominated")
df.loc[T == 0, "Dominance"] = np.nan  # mark undefined dominance

# -------------------------------
# Compute CV per cluster
# -------------------------------
group_cols = ["Jlf", "mu", "PP", "rep", "Cluster ID", "Dominance"]
cv_records = []

for group_key, cluster_df in df.groupby(group_cols):
    jlf, mu, pp, rep, cluster_id, dom_type = group_key
    cluster_sizes = cluster_df["Total Cells"].dropna().values
    time_points = len(cluster_sizes)

    if time_points == 0 or np.mean(cluster_sizes) == 0:
        mean_size = 0
        std_size = 0
        cv = np.nan
    else:
        mean_size = np.mean(cluster_sizes)
        std_size = np.std(cluster_sizes)
        cv = std_size / mean_size

    cv_records.append({
        "Jlf": jlf,
        "mu": mu,
        "PP": pp,
        "rep": rep,
        "Cluster ID": cluster_id,
        "Dominance": dom_type,
        "CV": cv if np.isfinite(cv) else np.nan,
        "Mean Size": mean_size,
        "Std Size": std_size,
        "Time Points": time_points
    })

# Create DataFrame
cv_df = pd.DataFrame(cv_records)

# -------------------------------
# Aggregate: mean CV by dominance
# -------------------------------
stability_summary = cv_df.groupby(["Jlf", "mu", "PP", "rep", "Dominance"]).agg({
    "CV": "mean",
    "Mean Size": "mean",
    "Std Size": "mean",
    "Time Points": "mean",   # Optional: shows avg lifespan of LD vs FD clusters
    "Cluster ID": "count"    # Number of clusters in each dominance category
}).rename(columns={"Cluster ID": "Num Clusters"}).reset_index()

# Optional: Pivot for easy comparison (LD vs FD per run)
pivot_df = stability_summary.pivot(
    index=["Jlf", "mu", "PP", "rep"],
    columns="Dominance",
    values="CV"
).reset_index()

# Save outputs
cv_per_cluster_path = "../../Results/Cluster/ClusterStability_perCluster.csv"
summary_path = "../../Results/Cluster/ClusterStability_by_Dominance.csv"

cv_df.to_csv(cv_per_cluster_path, index=False)
stability_summary.to_csv(summary_path, index=False)

print(f"Saved per-cluster CVs to: {cv_per_cluster_path}")
print(f"Saved dominance-wise CV summary to: {summary_path}")


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_80865/3144707752.py:6: DtypeWarning: Columns (11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Saved per-cluster CVs to: ../../Results/Cluster/ClusterStability_perCluster.csv
Saved dominance-wise CV summary to: ../../Results/Cluster/ClusterStability_by_Dominance.csv


# Each Cluster Data

In [39]:
import os
import math
import pandas as pd
import numpy as np

# -----------------------------
# Configuration
# -----------------------------
BEST_BY = "clusters"  # <-- "clusters" OR "persistence"
# BEST_BY = "persistence"

CLUSTERS_FILE       = "../../Results/Data/Clusters/ClusterComposition.csv"
BEST_FILE_IN        = None  # "../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv"   # optional override
PARA_SUMMARY_OUT    = "../../Results/Data/Clusters/clusters_per_parameter.csv"
BEST_FILE_OUT       = "../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv"
DETAIL_OUT          = "../../Results/Data/Clusters/best_clusters_detailed.csv"
STABILITY_TIME_OUT  = "../../Results/Data/Clusters/cluster_stability_time.csv"

DOMINANCE_THRESHOLD = 60.0  # percent

# -----------------------------
# Helpers
# -----------------------------
def safe_pct(numer, denom):
    try:
        numer = float(numer); denom = float(denom)
        return 0.0 if denom <= 0 else 100.0 * numer / denom
    except Exception:
        return 0.0

def euclid_dxdy(x0, y0, x1, y1):
    try:
        return math.hypot(float(x1) - float(x0), float(y1) - float(y0))
    except Exception:
        return 0.0

def dominance_label(leader_pct, threshold=DOMINANCE_THRESHOLD):
    return "Leader-dominated" if leader_pct >= threshold else "Follower-dominated"

def select_best_pp(clusters_per_param, first_pos, mode):
    """Select best PP per (Jlf, mu) based on chosen mode."""
    tmp = clusters_per_param.merge(first_pos, on=["Jlf", "mu", "PP"], how="left")
    if mode == "clusters":
        sort_cols = ["Jlf", "mu", "NumClusters", "AvgPersistence", "first_index"]
        ascending = [ True,  True,  False,        False,            True]
    else:  # "persistence"
        sort_cols = ["Jlf", "mu", "AvgPersistence", "NumClusters", "first_index"]
        ascending = [ True,  True,  False,           False,         True]
    tmp = tmp.sort_values(by=sort_cols, ascending=ascending, kind="stable")
    return (
        tmp.groupby(["Jlf", "mu"], as_index=False)
           .first()[["Jlf", "mu", "PP", "NumClusters", "AvgPersistence"]]
    )

def size_cv(series: pd.Series):
    """Coefficient of variation of a size time series (std/mean)."""
    if series.size < 2:
        return np.nan
    m = series.mean()
    if m is None or m <= 0:
        return np.nan
    s = series.std()  # sample std (ddof=1)
    return s / m

def phenotypic_index(L, F):
    """PI = |L-F| / (L+F)."""
    try:
        L = float(L); F = float(F)
        T = L + F
        if T <= 0:
            return np.nan
        return abs(L - F) / T
    except Exception:
        return np.nan

def shannon_entropy(L, F):
    """H = -pL log2 pL - pF log2 pF, with 0*log 0 := 0."""
    try:
        L = float(L); F = float(F)
        T = L + F
        if T <= 0:
            return np.nan
        pL = L / T
        pF = F / T
        termL = -pL * np.log2(pL) if pL > 0 else 0.0
        termF = -pF * np.log2(pF) if pF > 0 else 0.0
        return termL + termF
    except Exception:
        return np.nan

# -----------------------------
# Load and preprocess data
# -----------------------------
if not os.path.exists(CLUSTERS_FILE):
    raise FileNotFoundError(f"Cannot find required file: {CLUSTERS_FILE}")

df = pd.read_csv(CLUSTERS_FILE, low_memory=False)

required_cols = [
    "MCS", "Jlf", "mu", "PP", "Cluster ID",
    "Leader Cells", "Follower Cells", "Total Cells",
    "Centroid_X", "Centroid_Y"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {CLUSTERS_FILE}: {missing}")

clusters_df = df[df["Total Cells"] > 0].copy()
for col in ["Jlf", "mu", "PP", "MCS", "Cluster ID",
            "Leader Cells", "Follower Cells", "Total Cells",
            "Centroid_X", "Centroid_Y"]:
    clusters_df[col] = pd.to_numeric(clusters_df[col], errors="coerce")
clusters_df = clusters_df.dropna(subset=["Jlf", "mu", "PP", "Cluster ID", "MCS"])

# -----------------------------
# Stability across time (CV_t) and summaries
# -----------------------------
# Per (Jlf, mu, PP, rep, MCS): mean and std across clusters present at that time
cv_time = (
    clusters_df.groupby(["Jlf","mu","PP","rep","MCS"])["Total Cells"]
    .agg(mean_size_t="mean", std_size_t="std", n_clusters_t="count")
    .reset_index()
)

# CV_t = std / mean (only meaningful if >=2 clusters and mean > 0)
cv_time["CV_t"] = np.where(
    (cv_time["n_clusters_t"] >= 2) & (cv_time["mean_size_t"] > 0),
    cv_time["std_size_t"] / cv_time["mean_size_t"],
    np.nan
)

# Save per-time stability table
cv_time.to_csv(STABILITY_TIME_OUT, index=False)

# Summarize CV per replicate
cv_rep = (
    cv_time.groupby(["Jlf","mu","PP","rep"], dropna=False)["CV_t"]
    .agg(MeanCV_over_time="mean", MedianCV_over_time="median", Timepoints="count")
    .reset_index()
)

# Summarize CV per parameter (across replicates)
cv_param = (
    cv_rep.groupby(["Jlf","mu","PP"], dropna=False)
    .agg(MeanCV_over_time=("MeanCV_over_time","mean"),
         MedianCV_over_time=("MedianCV_over_time","median"),
         Reps=("rep","nunique"))
    .reset_index()
)

# -----------------------------
# Parameter-level summary (NumClusters, AvgPersistence) + merge CV summaries
# -----------------------------
# Persistence per cluster = number of distinct MCS for that cluster
persistence_per_cluster = (
    clusters_df.groupby(["Jlf", "mu", "PP", "Cluster ID"], as_index=False)["MCS"]
    .nunique().rename(columns={"MCS": "PersistenceTime"})
)
clusters_per_param = (
    persistence_per_cluster
    .groupby(["Jlf", "mu", "PP"], as_index=False)
    .agg(NumClusters=("Cluster ID", "nunique"),
         AvgPersistence=("PersistenceTime", "mean"))
)

# Merge CV summaries into parameter table
clusters_per_param = clusters_per_param.merge(cv_param, on=["Jlf","mu","PP"], how="left")

# First appearance index for tie-breaking and sorted param table
first_pos = (
    clusters_df.reset_index()
    .groupby(["Jlf", "mu", "PP"])["index"]
    .min()
    .reset_index()
    .rename(columns={"index": "first_index"})
)
clusters_per_param_sorted = (
    clusters_per_param
    .merge(first_pos, on=["Jlf", "mu", "PP"], how="left")
    .sort_values(by="first_index", kind="stable")
    .drop(columns=["first_index"])
)
clusters_per_param_sorted.to_csv(PARA_SUMMARY_OUT, index=False)

# -----------------------------
# Select best PP per (Jlf, mu)
# -----------------------------
if BEST_FILE_IN and os.path.exists(BEST_FILE_IN):
    best_df = pd.read_csv(BEST_FILE_IN, low_memory=False)
    if not {"NumClusters", "AvgPersistence"}.issubset(best_df.columns):
        best_df = best_df.merge(clusters_per_param, on=["Jlf", "mu", "PP"], how="left")
else:
    best_df = select_best_pp(clusters_per_param, first_pos, BEST_BY)
    best_df.to_csv(BEST_FILE_OUT, index=False)

for col in ["Jlf", "mu", "PP", "NumClusters", "AvgPersistence"]:
    if col in best_df.columns:
        best_df[col] = pd.to_numeric(best_df[col], errors="coerce")

# -----------------------------
# Build detailed cluster list (with per-cluster stats, PI, Entropy)
# -----------------------------
rows = []
clusters_idx = clusters_df.set_index(["Jlf", "mu", "PP"]).sort_index()

for rec in best_df[["Jlf", "mu", "PP"]].drop_duplicates().itertuples(index=False):
    jlf_val, mu_val, pp_val = rec
    try:
        subset = clusters_idx.loc[(jlf_val, mu_val, pp_val)].reset_index()
    except KeyError:
        continue
    for cid, g in subset.groupby("Cluster ID"):
        if len(g) == 0:
            continue
        g = g.sort_values("MCS", kind="stable")

        # Persistence and time range
        persistence = g["MCS"].nunique()
        start_time  = int(g["MCS"].min())
        end_time    = int(g["MCS"].max())
        time_span   = f"{start_time}-{end_time}"

        # Per-cluster size stats over its lifetime
        mean_size_over_time = g["Total Cells"].mean()
        std_size_over_time  = g["Total Cells"].std()  # sample std (ddof=1)
        cluster_size_cv     = size_cv(g["Total Cells"])

        # Time-averaged composition metrics (PI and Shannon entropy)
        pi_series = g.apply(lambda r: phenotypic_index(r["Leader Cells"], r["Follower Cells"]), axis=1)
        H_series  = g.apply(lambda r: shannon_entropy(r["Leader Cells"], r["Follower Cells"]), axis=1)
        PI_mean_over_time = float(np.nanmean(pi_series)) if pi_series.size else np.nan
        Entropy_mean_over_time = float(np.nanmean(H_series)) if H_series.size else np.nan

        # Start & end centroid
        start_x, start_y = g.iloc[0][["Centroid_X", "Centroid_Y"]]
        end_x, end_y     = g.iloc[-1][["Centroid_X", "Centroid_Y"]]
        displacement = euclid_dxdy(start_x, start_y, end_x, end_y)
        velocity     = displacement / persistence if persistence > 0 else 0.0

        # Final snapshot for composition/centroid
        last = g.iloc[-1]
        leader_cells   = pd.to_numeric(last.get("Leader Cells",   np.nan), errors="coerce")
        follower_cells = pd.to_numeric(last.get("Follower Cells", np.nan), errors="coerce")
        total_cells    = pd.to_numeric(last.get("Total Cells",    np.nan), errors="coerce")
        centroid_x     = pd.to_numeric(last.get("Centroid_X",     np.nan), errors="coerce")
        centroid_y     = pd.to_numeric(last.get("Centroid_Y",     np.nan), errors="coerce")
        leader_pct = safe_pct(leader_cells, total_cells)
        dom = dominance_label(leader_pct, DOMINANCE_THRESHOLD)

        # Final-snapshot PI & Entropy
        PI_final       = phenotypic_index(leader_cells, follower_cells)
        Entropy_final  = shannon_entropy(leader_cells, follower_cells)

        rows.append({
            "Jlf": jlf_val,
            "mu": mu_val,
            "PP": pp_val,
            "Cluster ID": cid,
            "Leader Cells": leader_cells,
            "Follower Cells": follower_cells,
            "Total Cells": total_cells,
            "Leader %": leader_pct,
            "Dominance": dom,
            "Centroid_X": centroid_x,
            "Centroid_Y": centroid_y,
            "PersistenceTime": persistence,
            "TimeRange": time_span,                     # start-stop as "start-end"
            "Centroid_Displacement": displacement,
            "Velocity": velocity,
            "MeanSize_over_time": mean_size_over_time,
            "StdSize_over_time": std_size_over_time,
            "ClusterSizeCV_over_time": cluster_size_cv,
            "Phenotypic Index": PI_final,
            "Entropy": Entropy_final
            # "PI_mean_over_time": PI_mean_over_time,
            # "Entropy_mean_over_time": Entropy_mean_over_time
        })

detailed = pd.DataFrame(rows)
if not detailed.empty:
    detailed = detailed.sort_values(
        by=["Jlf", "mu", "PP", "Cluster ID", "PersistenceTime"],
        ascending=[True, True, True, True, False],
        kind="stable"
    )
detailed.to_csv(DETAIL_OUT, index=False)

used_best = BEST_FILE_IN if (BEST_FILE_IN and os.path.exists(BEST_FILE_IN)) else BEST_FILE_OUT
print(f"[BEST_BY: {BEST_BY}]")
print(f"Saved per-time stability: {STABILITY_TIME_OUT}")
print(f"Saved parameter summary: {PARA_SUMMARY_OUT}")
print(f"Saved best (Jlf, mu, PP): {used_best}")
print(f"Saved detailed clusters: {DETAIL_OUT}")


[BEST_BY: clusters]
Saved per-time stability: ../../Results/Data/Clusters/cluster_stability_time.csv
Saved parameter summary: ../../Results/Data/Clusters/clusters_per_parameter.csv
Saved best (Jlf, mu, PP): ../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv
Saved detailed clusters: ../../Results/Data/Clusters/best_clusters_detailed.csv


## Merging/Splitting

In [34]:
import pandas as pd
import numpy as np

# --- Tunable thresholds ---
D_THRESH = 5.0     # max centroid distance to count as same
S_DIFF_MAX = 0.30  # max relative size change
DOWNSAMPLE_STEP = 1  # set to >1 to skip frames (e.g., 2 keeps every other MCS)

# Paths
clusters_path = "../../Results/Data/Clusters/ClusterComposition.csv"
best_path = "../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv"  # optional

# Load
df = pd.read_csv(clusters_path, low_memory=False)
df = df[df["Total Cells"] > 0].copy()

num_cols = ["MCS","Jlf","mu","PP","rep","Cluster ID","Total Cells","Centroid_X","Centroid_Y"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=num_cols)

# Optional: focus only on selected parameter combos to speed up
try:
    best_df = pd.read_csv(best_path)
    for c in ["Jlf","mu","PP"]:
        best_df[c] = pd.to_numeric(best_df[c], errors="coerce")
    df = df.merge(best_df[["Jlf","mu","PP"]].drop_duplicates(), on=["Jlf","mu","PP"], how="inner")
except Exception:
    pass

events = []

for (jlf, mu, pp, rep), g in df.groupby(["Jlf","mu","PP","rep"]):
    g = g.sort_values(["MCS","Cluster ID"])
    times = sorted(g["MCS"].unique())
    if DOWNSAMPLE_STEP > 1:
        times = times[::DOWNSAMPLE_STEP]
    time_groups = {t: x for t, x in g.groupby("MCS")}
    
    for i in range(len(times)-1):
        t, t_next = times[i], times[i+1]
        if t not in time_groups or t_next not in time_groups:
            continue
        
        prev_df = time_groups[t][["Cluster ID","Total Cells","Centroid_X","Centroid_Y"]].reset_index(drop=True)
        next_df = time_groups[t_next][["Cluster ID","Total Cells","Centroid_X","Centroid_Y"]].reset_index(drop=True)
        if prev_df.empty or next_df.empty:
            continue
        
        # Vectorized pairing between prev and next
        P = prev_df.shape[0]; N = next_df.shape[0]
        p_ids = prev_df["Cluster ID"].to_numpy(dtype=int)
        n_ids = next_df["Cluster ID"].to_numpy(dtype=int)
        p_xy  = prev_df[["Centroid_X","Centroid_Y"]].to_numpy(float)
        n_xy  = next_df[["Centroid_X","Centroid_Y"]].to_numpy(float)
        p_sz  = prev_df["Total Cells"].to_numpy(float).reshape(P,1)
        n_sz  = next_df["Total Cells"].to_numpy(float).reshape(1,N)
        
        diff = p_xy[:,None,:] - n_xy[None,:,:]           # (P,N,2)
        dist = np.sqrt((diff**2).sum(axis=2))            # (P,N)
        denom = np.maximum(p_sz, n_sz)
        size_diff = np.abs(n_sz - p_sz) / np.maximum(denom, 1.0)
        
        match_mask = (dist <= D_THRESH) & (size_diff <= S_DIFF_MAX)  # (P,N)
        
        # Splits: row has >=2 matches
        row_counts = match_mask.sum(axis=1)
        split_rows = np.where(row_counts >= 2)[0]
        for r in split_rows:
            js = np.where(match_mask[r])[0]
            events.append({
                "type": "split",
                "Jlf": jlf, "mu": mu, "PP": pp, "rep": rep,
                "time_from": t, "time_to": t_next,
                "from_cluster": int(p_ids[r]),
                "to_clusters": [int(n_ids[j]) for j in js],
                "avg_dist": float(dist[r, js].mean()),
                "avg_size_diff_frac": float(size_diff[r, js].mean()),
                "D_THRESH": D_THRESH, "S_DIFF_MAX": S_DIFF_MAX,
                "DOWNSAMPLE_STEP": DOWNSAMPLE_STEP
            })
        
        # Merges: column has >=2 matches
        col_counts = match_mask.sum(axis=0)
        merge_cols = np.where(col_counts >= 2)[0]
        for c in merge_cols:
            is_ = np.where(match_mask[:, c])[0]
            events.append({
                "type": "merge",
                "Jlf": jlf, "mu": mu, "PP": pp, "rep": rep,
                "time_from": t, "time_to": t_next,
                "from_clusters": [int(p_ids[i]) for i in is_],
                "to_cluster": int(n_ids[c]),
                "avg_dist": float(dist[is_, c].mean()),
                "avg_size_diff_frac": float(size_diff[is_, c].mean()),
                "D_THRESH": D_THRESH, "S_DIFF_MAX": S_DIFF_MAX,
                "DOWNSAMPLE_STEP": DOWNSAMPLE_STEP
            })

events_df = pd.DataFrame(events)
events_df.to_csv("../../Results/Data/Clusters/cluster_merge_split_centroid.csv", index=False)
events_df.head()


,type,Jlf,mu,PP,rep,time_from,time_to,from_clusters,to_cluster,avg_dist,avg_size_diff_frac,D_THRESH,S_DIFF_MAX,DOWNSAMPLE_STEP,from_cluster,to_clusters
0,merge,0,27,0.0,9,650,660,"[3, 5]",4.0,3.015953,0.1000,5.0,0.3,1,NaN,NaN
1,merge,3,27,0.2,2,670,680,"[1, 4]",4.0,2.856476,0.0625,5.0,0.3,1,NaN,NaN
2,split,3,27,0.2,6,640,650,NaN,NaN,2.686067,0.0000,5.0,0.3,1,9.0,"[9, 11]"
3,split,3,27,0.2,6,650,660,NaN,NaN,1.994674,0.0000,5.0,0.3,1,11.0,"[9, 11]"
4,merge,3,27,0.2,6,650,660,"[9, 11]",9.0,2.406674,0.0000,5.0,0.3,1,NaN,NaN
